### 1. Import required libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import joblib
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold

sns.set_theme(style="whitegrid")
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

### 2. Load and concatenate datasets

In [ ]:
file_pattern = "../data/raw/*.csv"
csv_files = glob.glob(file_pattern)

print(f"Found {len(csv_files)} CSV files. Loading data, this might take a moment...")

if len(csv_files) == 0:
    raise FileNotFoundError("Error! CSV files not found!")

df_list = [pd.read_csv(file) for file in csv_files]
df = pd.concat(df_list, ignore_index=True)

print(f"Data loaded successfully!")
print(f"Dataset Shape: {df.shape[0]} rows and {df.shape[1]} columns.")

### 3. Basic Data Cleaning

In [ ]:
df.columns = df.columns.str.strip()
print("Column names stripped of whitespace.")

initial_nans = df.isna().sum().sum()
print(f"Initial NaN values: {initial_nans}")

df.replace([np.inf, -np.inf], np.nan, inplace=True)
total_nans_after_inf = df.isna().sum().sum()
print(f"Total missing values after replacing Infs with NaNs: {total_nans_after_inf}")

df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Data shape after cleaning: {df.shape}")

### 4. Exploratory Data Analysis - Label Distribution

In [ ]:
label_counts = df['Label'].value_counts()

print("Class Distribution:")
print(label_counts)

plt.figure(figsize=(12, 6))
sns.barplot(y=label_counts.index, x=label_counts.values, palette='viridis')
plt.title('Distribution of Network Traffic Labels (Log Scale)')
plt.xlabel('Number of Samples')
plt.ylabel('Label')
plt.xscale('log')
plt.show()

### 5. Target Engineering - Binary Classification

In [ ]:
df['Binary_Label'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

print("Binary Classification Distribution:")
print(df['Binary_Label'].value_counts())

X = df.drop(['Label', 'Binary_Label'], axis=1)
y = df['Binary_Label']

print(f"Features shape (X): {X.shape}")
print(f"Target shape (y): {y.shape}")

### 6. Train / Validation / Test Split

In [ ]:
# First split: 70% Training, 30% Temporary (Validation + Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Second split: Divide the temporary set into two equal parts (15% Validation, 15% Test)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Training set:   X={X_train.shape}, y={y_train.shape}")
print(f"Validation set: X={X_val.shape}, y={y_val.shape}")
print(f"Test set:       X={X_test.shape},  y={y_test.shape}")

### 7. Feature Normalization (Standardization)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("Data normalization complete.")

### 8. Feature Selection using Random Forest Importance

In [ ]:
print("Training a baseline Random Forest to determine feature importances...")
start_time = time.time()

rf_selector = RandomForestClassifier(
    n_estimators=500,     
    max_depth=30,          
    class_weight='balanced', 
    min_samples_split=5,     
    random_state=42, 
    n_jobs=-1                
)

rf_selector.fit(X_train_scaled, y_train)

selection_time = time.time() - start_time
print(f"Feature evaluation completed in {selection_time / 60:.2f} minutes!")

importances = rf_selector.feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(20), palette='mako')
plt.title('Top 20 Most Important Features')
plt.xlabel('Importance Score')
plt.ylabel('Feature Name')
plt.show()

TOP_FEATURES_COUNT = 40
best_features = feature_importance_df['Feature'].head(TOP_FEATURES_COUNT).tolist()

X_train_final = X_train_scaled[best_features]
X_val_final = X_val_scaled[best_features]
X_test_final = X_test_scaled[best_features]

print(f"Feature Selection complete. Retained top {TOP_FEATURES_COUNT} features.")
print(f"Final Training shape: {X_train_final.shape}")

### 9. Save fully processed and selected datasets

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

print("Saving processed datasets. Please wait...")

joblib.dump((X_train_final, y_train), '../data/processed/train_data.joblib')
joblib.dump((X_val_final, y_val), '../data/processed/val_data.joblib')
joblib.dump((X_test_final, y_test), '../data/processed/test_data.joblib')

print("Success! Fully processed datasets are safely stored in 'data/processed/'")